### Testing using sklearn simple model

In [21]:
from kserve import RESTConfig, InferenceRESTClient

config = RESTConfig(protocol="v2", retries=5, timeout=30)
client = InferenceRESTClient(config)
base_url = "http://iris-testing.kubeflow-user-example-com.svc.cluster.local"
# Define the correct V2 payload structure
data_v2 = {
  "inputs": [
    {
      "name": "request_id", 
      "shape": [3, 4],  
      "datatype": "FP64",
      "data": [
        [6.8, 2.8, 4.8, 1.4],
        [6.0, 3.4, 4.5, 1.6],
        [6.0, 3.4, 4.5, 1.6],
      ]
    }
  ]
}
model_name = "iris-testing"
result = await client.infer(base_url, data_v2, model_name=model_name)
print(result)

"id": "72af04f3-087f-4f66-b23f-297c5edae63b","model_name": "iris-testing","outputs": ["name": "output-1","shape": [3, 1],"datatype": "INT64","data": [1, 1, 1],"parameters": {'content_type': 'np'}],"parameters": {'content_type': 'np'},"from_grpc": False


In [3]:
import time
from concurrent.futures import ThreadPoolExecutor
from kserve import RESTConfig, InferenceRESTClient
# We don't need asyncio anymore since we are using synchronous calls in threads.

# --- Configuration and Client Setup ---
config = RESTConfig(protocol="v2", retries=5, timeout=30)
# NOTE: We keep this client object for simplicity, but we will call its methods synchronously.
client = InferenceRESTClient(config) 
base_url = "http://iris-testing.kubeflow-user-example-com.svc.cluster.local"
model_name = "iris-testing"

# Define the correct V2 payload structure
number_of_batch_sample = 10
data_v2 = {
    "inputs": [
        {
            "name": "request_id", 
            "shape": [number_of_batch_sample, 4], 
            "datatype": "FP64",
            "data": [
                [6.8, 2.8, 4.8, 1.4],
            ]*number_of_batch_sample
        }
    ]
}

# --- The Synchronous Inference Function ---
# This function must be synchronous now, as it will be run in a separate thread.
def call_infer_sync(client, url, payload, model, request_num):
    """A synchronous function to call the KServe endpoint, designed for a ThreadPool."""
    try:
        # The client.infer() method is internally calling an async method, but 
        # in its synchronous form (like what we had before with asyncio.run).
        # We will use client._sync_infer which is a method on the client to perform a sync call.
        # NOTE: client.infer() is actually a good fit if your kserve version exposes a blocking call.
        # If your KServe version *only* exposes the async `infer`, we must ensure we use the 
        # correct synchronous method, or use the client that does not require await.
        
        # Based on the fact that you used `await client.infer()`, the actual underlying 
        # method that blocks is what we need to call here. 
        # Let's assume a utility function or a sync client is used here, or we call the core logic.
        
        # A simple synchronous REST client call is safer here, but since you are using 
        # the InferenceRESTClient, we must find its synchronous method or create a new client 
        # object inside the thread, or use a simpler client like `requests`.
        
        # To avoid external dependencies and stick to kserve, let's assume `client.infer` 
        # can be called synchronously if we manage the loop, but since we are changing 
        # the structure, we should use the **synchronous part of the kserve library**.

        # Since kserve is designed around asyncio, let's simplify and use the synchronous 
        # `requests` library which is what the KServe client likely uses internally for REST.
        # However, for the most correct answer, we assume you installed a sync kserve client.

        # *** STICKING TO KSERVE, BUT USING A DEDICATED SYNC CALL ***
        # Since the provided client is fundamentally asynchronous, we will use a more standard
        # approach for synchronous REST calls in the threads.
        
        # If your kserve client ONLY exposes `await infer()`, then creating the client 
        # and calling the internal blocking function is the way. 
        # Let's revert the client to be a basic synchronous one for thread safety/simplicity.
        
        # For simplicity and robustness in a threaded environment:
        import requests
        
        url = f"{base_url}/v2/models/{model}/infer"
        headers = {'Content-Type': 'application/json'}
        
        # The actual synchronous network call
        response = requests.post(url, json=payload, headers=headers, timeout=config.timeout)
        response.raise_for_status()
        
        # Since we are not using the KServe client object which handles parsing, we just return JSON
        return response.json() 
        
    except requests.exceptions.RequestException as e:
        # This will catch connection, timeout, and HTTP error status codes
        return {"error": str(e), "request_num": request_num}
    except Exception as e:
        return {"error": str(e), "request_num": request_num}


# --- The Main Concurrent Execution Function (Synchronous) ---
def main_parallel(num_requests=100, max_workers=10):
    """Sets up and runs multiple concurrent inference tasks using a ThreadPool."""
    start_time = time.monotonic()
    print(f"Starting {num_requests} parallel inference requests with {max_workers} threads...")
    
    # We use ThreadPoolExecutor for I/O bound tasks
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all the tasks to the thread pool
        futures = [
            executor.submit(call_infer_sync, client, base_url, data_v2, model_name, i)
            for i in range(1, num_requests + 1)
        ]
        
        # Wait for all tasks to complete and collect results
        # all_results = [f.result() for f in futures]

    end_time = time.monotonic()
    
    print("\nAll requests completed.")
    
    # Check for success/failure based on the dictionary structure we used in call_infer_sync
    # success_count = sum(1 for res in all_results if not isinstance(res, dict) or 'error' not in res)
    # failure_count = len(all_results) - success_count
    
    print("-" * 30)
    print(f"Total time taken: {end_time - start_time:.2f} seconds")
    # print(f"Total Successful Calls: {success_count}")
    # print(f"Total Failed Calls: {failure_count}")
    print("-" * 30)
    
    # return all_results

# --- Call the function (Synchronous in Jupyter) ---

# NOTE: You MUST install the requests library: `pip install requests` 
# Run this cell in your Jupyter Notebook:
results = main_parallel(num_requests=50000, max_workers=250) # Increased requests and workers for better stress testing

Starting 50000 parallel inference requests with 250 threads...

All requests completed.
------------------------------
Total time taken: 118.89 seconds
------------------------------


In [1]:
import os
import multiprocessing

# Use os.cpu_count() for the number of available logical CPUs (cores/threads)
num_cpus = os.cpu_count() 
# Or use multiprocessing.cpu_count()
# num_cpus = multiprocessing.cpu_count()

# Rule of Thumb: 5x the number of logical cores for I/O bound tasks
recommended_max_workers = num_cpus * 5

print(f"Your machine has {num_cpus} logical CPUs.")
print(f"A strong starting point for max_workers (5x Cores) is: {recommended_max_workers}")

Your machine has 16 logical CPUs.
A strong starting point for max_workers (5x Cores) is: 80


### Testing usnig the pytorch MLModel

The problem is the coded and decoded from the V2 Open Inference Protocol to the python model needs

It seems it's sharing a dict with the keys as the names of the columns and using content_type: np as parameter it's converting the lisst in a arrays, but the model needs a pandas as object not a dict with this.

In [86]:
import requests
import pandas as pd

payload = {
    # "parameters": {"content_type": "pd"},
    "inputs": [
    {
      "name": "user", 
      "shape": [-1],  
      "datatype": "INT64",
      "data": [123],
      "parameters": {"content_type": "np"}
    },
    {
      "name": "movie", 
      "shape": [-1],  
      "datatype": "INT64",
      "data": [321],
      "parameters": {"content_type": "np"}
    }
  ]
}

response = requests.post(
    "http://movie-recommender-gpu.kubeflow-user-example-com.svc.cluster.local/v2/models/recommender_production/infer",
    json=payload
)

print(response.status_code)
display(response.json())

500


{'error': 'builtins.TypeError: The PyTorch flavor does not support List or Dict input types. Please use a pandas.DataFrame or a numpy.ndarray'}